### Previsão de Demanda com Random Forest Regressor

Este notebook tem como objetivo prever o volume total de pedidos esperado até o fim do dia/turno, utilizando dados históricos de pedidos das Squads 2 e 3.

A abordagem utiliza Engenharia de Features para transformar os dados transacionais em variáveis numéricas, como pedidos acumulados, receita acumulada, ticket médio, hora do pedido, dia da semana e percentual do dia decorrido.

O modelo escolhido foi o Random Forest Regressor, pois ele trabalha bem com dados tabulares e consegue capturar relações não lineares no comportamento das vendas.

In [0]:
import os
import pandas as pd
import numpy as np

from dotenv import load_dotenv

from pyspark.sql import functions as F
from pyspark.sql.window import Window

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

Célula 3 - Conexão SQL Serve

In [0]:
load_dotenv()

sql_host = os.getenv("SQL_HOST")
sql_database = os.getenv("SQL_DATABASE")
sql_username = os.getenv("SQL_USERNAME")
sql_password = os.getenv("SQL_PASSWORD")

jdbc_url = (
    f"jdbc:sqlserver://{sql_host}:1433;"
    f"databaseName={sql_database};"
    "encrypt=true;"
    "trustServerCertificate=false;"
    "hostNameInCertificate=*.database.windows.net;"
    "loginTimeout=30;"
)

connection_properties = {
    "user": sql_username,
    "password": sql_password,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}


Célula 4 - Ler Pedidos Squad 2 e Squad 3

In [0]:
query_squad2 = """
(
    SELECT
        id_pedido,
        id_cliente,
        id_endereco_entrega,
        dt_pedido,
        status_pedido,
        valor_total,
        valor_frete,
        metodo_pagamento,
        dt_ultima_atualizacao_status
    FROM squad2.ecommerce_pedidos
) AS pedidos_squad2
"""

query_squad3 = """
(
    SELECT
        id_pedido,
        id_cliente,
        id_endereco_entrega,
        dt_pedido,
        status_pedido,
        valor_total,
        valor_frete,
        metodo_pagamento,
        dt_ultima_atualizacao_status
    FROM squad3.ecommerce_pedidos
) AS pedidos_squad3
"""

df_pedidos_squad2 = spark.read.jdbc(
    url=jdbc_url,
    table=query_squad2,
    properties=connection_properties
)

df_pedidos_squad3 = spark.read.jdbc(
    url=jdbc_url,
    table=query_squad3,
    properties=connection_properties
)

display(df_pedidos_squad2.limit(10))
display(df_pedidos_squad3.limit(10))

 Juntar Squad 2 + Squad 3

In [0]:
df_pedidos_squad2_padrao = (
    df_pedidos_squad2
    .withColumn("origem_squad", F.lit("squad2"))
)

df_pedidos_squad3_padrao = (
    df_pedidos_squad3
    .withColumn("origem_squad", F.lit("squad3"))
)

df_pedidos_unificado = df_pedidos_squad2_padrao.unionByName(
    df_pedidos_squad3_padrao,
    allowMissingColumns=True
)

display(df_pedidos_unificado.limit(50))

print("Total Squad 2:", df_pedidos_squad2_padrao.count())
print("Total Squad 3:", df_pedidos_squad3_padrao.count())
print("Total Unificado:", df_pedidos_unificado.count())

Engenharia de Features

In [0]:
df_base = (
    df_pedidos_unificado
    .withColumn("dt_pedido", F.to_timestamp("dt_pedido"))
    .withColumn("data_pedido", F.to_date("dt_pedido"))
    .withColumn("hora_pedido", F.hour("dt_pedido"))
    .withColumn("dia_semana", F.dayofweek("dt_pedido"))
    .withColumn("valor_total", F.col("valor_total").cast("double"))
)

df_hora = (
    df_base
    .groupBy("data_pedido", "hora_pedido", "dia_semana")
    .agg(
        F.countDistinct("id_pedido").alias("pedidos_hora"),
        F.sum("valor_total").alias("receita_hora"),
        F.avg("valor_total").alias("ticket_medio_hora")
    )
)

janela_dia = (
    Window
    .partitionBy("data_pedido")
    .orderBy("hora_pedido")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

janela_total_dia = Window.partitionBy("data_pedido")

df_modelo_spark = (
    df_hora
    .withColumn("pedidos_acumulados", F.sum("pedidos_hora").over(janela_dia))
    .withColumn("receita_acumulada", F.sum("receita_hora").over(janela_dia))
    .withColumn("total_pedidos_dia", F.sum("pedidos_hora").over(janela_total_dia))
    .withColumn("tempo_restante_dia", F.lit(23) - F.col("hora_pedido"))
    .withColumn("percentual_dia_decorrido", (F.col("hora_pedido") + F.lit(1)) / F.lit(24))
)

display(df_modelo_spark)

Treinar Random Forest

In [0]:
df_modelo = df_modelo_spark.toPandas()
df_modelo = df_modelo.fillna(0)

features = [
    "hora_pedido",
    "dia_semana",
    "pedidos_hora",
    "receita_hora",
    "ticket_medio_hora",
    "pedidos_acumulados",
    "receita_acumulada",
    "tempo_restante_dia",
    "percentual_dia_decorrido"
]

X = df_modelo[features]
y = df_modelo["total_pedidos_dia"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

modelo_rf = RandomForestRegressor(
    n_estimators=150,
    max_depth=8,
    random_state=42,
    n_jobs=-1
)

modelo_rf.fit(X_train, y_train)

predicoes = modelo_rf.predict(X_test)

mae = mean_absolute_error(y_test, predicoes)
rmse = np.sqrt(mean_squared_error(y_test, predicoes))
r2 = r2_score(y_test, predicoes)

print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²: {r2:.2f}")

Ver Previsões

In [0]:
df_resultado = pd.DataFrame({
    "real_total_pedidos_dia": y_test.values,
    "previsao_total_pedidos_dia": predicoes
})

df_resultado["erro"] = (
    df_resultado["real_total_pedidos_dia"]
    - df_resultado["previsao_total_pedidos_dia"]
)

display(df_resultado.head(20))

Importância das Features

In [0]:
importancias = pd.DataFrame({
    "feature": features,
    "importancia": modelo_rf.feature_importances_
}).sort_values("importancia", ascending=False)

display(importancias)

cria a função prever_total_pedidos_novo

In [0]:
def prever_total_pedidos_novo(df_novo_spark):
    df_novo = df_novo_spark.toPandas()
    df_novo = df_novo.fillna(0)

    X_novo = df_novo[features]

    df_novo["previsao_total_pedidos_dia"] = modelo_rf.predict(X_novo)

    return df_novo

Criar Previsão

In [0]:
df_previsao_teste = prever_total_pedidos_novo(df_modelo_spark)

display(df_previsao_teste.head(10))

Calcular Erro Absoluto


In [0]:
df_previsao_teste["erro_absoluto"] = abs(
    df_previsao_teste["total_pedidos_dia"]
    - df_previsao_teste["previsao_total_pedidos_dia"]
)

display(
    df_previsao_teste[
        [
            "data_pedido",
            "hora_pedido",
            "pedidos_acumulados",
            "total_pedidos_dia",
            "previsao_total_pedidos_dia",
            "erro_absoluto"
        ]
    ].head(10)
)